# Data Preprocessing: Airline Passenger Satisfaction

**Dataset**: Airline Passenger Satisfaction  
**Objective**: Clean, transform, and prepare the dataset for machine learning model training  
**Output**: JSON configuration files documenting each preprocessing step

## 1. Import Required Libraries

In [1]:
# Import libraries and set up the environment
import pandas as pd
import numpy as np
import json
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)

# Set working directory to project root
while not os.path.exists('requirements.txt') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')

# Create report directory for preprocessing JSON outputs
reports_dir = "reports/preprocessing"
os.makedirs(reports_dir, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Preprocessing report directory: {reports_dir}")
print(f"All outputs will be saved as JSON files only.")

Working directory: c:\Users\finle\Git_projects\ML_Assingnment_2
Preprocessing report directory: reports/preprocessing
All outputs will be saved as JSON files only.


## 2. Load the Dataset

In [2]:
# Load the dataset
df = pd.read_csv("dataset/test.csv")
df_original = df.copy()

print("\n" + "="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nShape: {df.shape[0]:,} records × {df.shape[1]} features")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nFirst few records:")
df.head()


DATASET OVERVIEW

Shape: 98,904 records × 25 features

Column types:
Unnamed: 0                             int64
id                                     int64
Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service              

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [3]:
# Save dataset overview as JSON
dataset_overview = {
    "dataset_name": "Airline Passenger Satisfaction",
    "source_file": "dataset/test.csv",
    "total_records": int(df.shape[0]),
    "total_features": int(df.shape[1]),
    "column_names": df.columns.tolist(),
    "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
    "memory_usage_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
}

with open(f"{reports_dir}/dataset_overview.json", "w") as f:
    json.dump(dataset_overview, f, indent=2)

print(f"Saved: {reports_dir}/dataset_overview.json")

Saved: reports/preprocessing/dataset_overview.json


## 3. Explore the Dataset

In [4]:
# Data quality assessment
print("\n" + "="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

# Missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

print(f"\nMissing values per column:")
if missing_data.sum() == 0:
    print("  No missing values detected")
else:
    for col in missing_data[missing_data > 0].index:
        print(f"  {col}: {missing_data[col]} ({missing_percent[col]:.2f}%)")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Data types breakdown
print(f"\nData types breakdown:")
print(f"  Numeric columns: {len(df.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"  Categorical columns: {len(df.select_dtypes(include=['object']).columns)}")

# Descriptive statistics
print(f"\nDescriptive statistics:")
print(df.describe())


DATA QUALITY ASSESSMENT

Missing values per column:
  Arrival Delay in Minutes: 299 (0.30%)

Duplicate rows: 0

Data types breakdown:
  Numeric columns: 20
  Categorical columns: 5

Descriptive statistics:
          Unnamed: 0             id           Age  Flight Distance  \
count   98904.000000   98904.000000  98904.000000     98904.000000   
mean    52003.363464   64980.800524     39.371805      1189.646324   
std     30001.153118   37454.398194     15.114978       997.154483   
min         0.000000       1.000000      7.000000        31.000000   
25%     25987.750000   32590.750000     27.000000       414.000000   
50%     52026.500000   64970.500000     40.000000       842.000000   
75%     78020.250000   97428.250000     51.000000      1744.000000   
max    103903.000000  129880.000000     85.000000      4983.000000   

       Inflight wifi service  Departure/Arrival time convenient  \
count           98904.000000                       98904.000000   
mean                2.730719

In [5]:
# Save data quality report as JSON
data_quality_report = {
    "missing_values": {col: int(val) for col, val in missing_data.items() if val > 0},
    "missing_percentages": {col: round(float(val), 4) for col, val in missing_percent.items() if val > 0},
    "duplicate_rows": int(duplicates),
    "total_records": int(len(df)),
    "data_completeness_percent": round(float(100 - (missing_data.sum() / (len(df) * len(df.columns)) * 100)), 4),
    "columns_with_missing": [col for col in missing_data.index if missing_data[col] > 0],
    "numeric_columns": df.select_dtypes(include=['int64', 'float64']).columns.tolist(),
    "categorical_columns": df.select_dtypes(include=['object']).columns.tolist()
}

with open(f"{reports_dir}/data_quality_report.json", "w") as f:
    json.dump(data_quality_report, f, indent=2)

print(f"Saved: {reports_dir}/data_quality_report.json")

Saved: reports/preprocessing/data_quality_report.json


## 4. Handle Missing Values

In [6]:
# Step 1: Drop unnecessary columns
columns_before = df.columns.tolist()

df.drop(columns=['id', 'Unnamed: 0'], inplace=True, errors='ignore')

columns_after = df.columns.tolist()
columns_dropped = [col for col in columns_before if col not in columns_after]

print("\n" + "="*60)
print("COLUMN CLEANUP")
print("="*60)
print(f"\nColumns before: {len(columns_before)}")
print(f"Columns dropped: {columns_dropped}")
print(f"Columns after: {len(columns_after)}")

# Save column cleanup as JSON
column_cleanup = {
    "columns_before": columns_before,
    "columns_after": columns_after,
    "columns_dropped": columns_dropped,
    "remaining_feature_count": len(columns_after)
}

with open(f"{reports_dir}/column_cleanup.json", "w") as f:
    json.dump(column_cleanup, f, indent=2)

print(f"\nSaved: {reports_dir}/column_cleanup.json")


COLUMN CLEANUP

Columns before: 25
Columns dropped: ['Unnamed: 0', 'id']
Columns after: 23

Saved: reports/preprocessing/column_cleanup.json


In [7]:
# Step 2: Record missing values BEFORE imputation (will impute AFTER split to avoid data leakage)
missing_before = df.isnull().sum()

print("\n" + "="*60)
print("MISSING VALUE ASSESSMENT (pre-split)")
print("="*60)
print(f"\nMissing values detected:")
for col in missing_before[missing_before > 0].index:
    print(f"  {col}: {missing_before[col]} ({missing_before[col]/len(df)*100:.2f}%)")
print(f"\nNote: Imputation will be performed AFTER train-test split")
print(f"      to prevent data leakage (fit median on train only).")


MISSING VALUE ASSESSMENT (pre-split)

Missing values detected:
  Arrival Delay in Minutes: 299 (0.30%)

Note: Imputation will be performed AFTER train-test split
      to prevent data leakage (fit median on train only).


## 5. Encode Categorical Variables

In [8]:
# Step 3: Encode the target variable
print("\n" + "="*60)
print("TARGET VARIABLE ENCODING")
print("="*60)

class_distribution_before = df['satisfaction'].value_counts().to_dict()
print(f"\nBefore encoding:")
print(df['satisfaction'].value_counts())

encoding_map = {'neutral or dissatisfied': 0, 'satisfied': 1}
df['satisfaction'] = df['satisfaction'].map(encoding_map)

class_distribution_after = df['satisfaction'].value_counts().to_dict()
print(f"\nAfter encoding:")
print(df['satisfaction'].value_counts())

class_balance_ratio = round(float(df['satisfaction'].value_counts().min() / df['satisfaction'].value_counts().max()), 4)
print(f"\nClass balance ratio (minority/majority): {class_balance_ratio}")

# Save target encoding as JSON
target_encoding = {
    "target_column": "satisfaction",
    "encoding_map": encoding_map,
    "class_distribution_before": {str(k): int(v) for k, v in class_distribution_before.items()},
    "class_distribution_after": {str(k): int(v) for k, v in class_distribution_after.items()},
    "class_balance_ratio": class_balance_ratio
}

with open(f"{reports_dir}/target_encoding.json", "w") as f:
    json.dump(target_encoding, f, indent=2)

print(f"\nSaved: {reports_dir}/target_encoding.json")


TARGET VARIABLE ENCODING

Before encoding:
satisfaction
neutral or dissatisfied    56083
satisfied                  42821
Name: count, dtype: int64

After encoding:
satisfaction
0    56083
1    42821
Name: count, dtype: int64

Class balance ratio (minority/majority): 0.7635

Saved: reports/preprocessing/target_encoding.json


In [9]:
# Step 4: Separate features and target
X = df.drop(columns=['satisfaction'])
y = df['satisfaction']

print("\n" + "="*60)
print("FEATURE-TARGET SEPARATION")
print("="*60)
print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")

# Save feature-target split as JSON
feature_target_split = {
    "feature_count": int(X.shape[1]),
    "target_column": "satisfaction",
    "feature_columns": X.columns.tolist(),
    "X_shape": list(X.shape),
    "y_shape": list(y.shape)
}

with open(f"{reports_dir}/feature_target_split.json", "w") as f:
    json.dump(feature_target_split, f, indent=2)

print(f"\nSaved: {reports_dir}/feature_target_split.json")


FEATURE-TARGET SEPARATION

Features (X) shape: (98904, 22)
Target (y) shape: (98904,)
Feature columns: ['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

Saved: reports/preprocessing/feature_target_split.json


In [10]:
# Step 5: One-hot encode categorical features
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
features_before = X.columns.tolist()
total_features_before = len(features_before)

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

features_after = X.columns.tolist()
new_encoded_columns = [col for col in features_after if col not in features_before]

print("\n" + "="*60)
print("CATEGORICAL ENCODING (ONE-HOT)")
print("="*60)
print(f"\nOriginal categorical columns: {categorical_cols}")
print(f"New encoded columns: {new_encoded_columns}")
print(f"Features before encoding: {total_features_before}")
print(f"Features after encoding: {len(features_after)}")

# Save categorical encoding as JSON
categorical_encoding = {
    "encoding_method": "one_hot",
    "drop_first": True,
    "original_categorical_columns": categorical_cols,
    "new_encoded_columns": new_encoded_columns,
    "total_features_before": total_features_before,
    "total_features_after": len(features_after),
    "all_feature_columns_after_encoding": features_after
}

with open(f"{reports_dir}/categorical_encoding.json", "w") as f:
    json.dump(categorical_encoding, f, indent=2)

print(f"\nSaved: {reports_dir}/categorical_encoding.json")


CATEGORICAL ENCODING (ONE-HOT)

Original categorical columns: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
New encoded columns: ['Gender_Male', 'Customer Type_disloyal Customer', 'Type of Travel_Personal Travel', 'Class_Eco', 'Class_Eco Plus']
Features before encoding: 22
Features after encoding: 23

Saved: reports/preprocessing/categorical_encoding.json


## 6. Split Data into Training and Testing Sets

In [11]:
# Step 6: Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"\nTrain satisfaction ratio: {y_train.mean():.4f}")
print(f"Test satisfaction ratio:  {y_test.mean():.4f}")

# Save train-test split as JSON
train_test_split_info = {
    "test_size": 0.2,
    "random_state": 42,
    "stratify": True,
    "X_train_shape": list(X_train.shape),
    "X_test_shape": list(X_test.shape),
    "y_train_shape": list(y_train.shape),
    "y_test_shape": list(y_test.shape),
    "train_satisfaction_ratio": round(float(y_train.mean()), 4),
    "test_satisfaction_ratio": round(float(y_test.mean()), 4)
}

with open(f"{reports_dir}/train_test_split.json", "w") as f:
    json.dump(train_test_split_info, f, indent=2)

print(f"\nSaved: {reports_dir}/train_test_split.json")


TRAIN-TEST SPLIT

X_train shape: (79123, 23)
X_test shape:  (19781, 23)
y_train shape: (79123,)
y_test shape:  (19781,)

Train satisfaction ratio: 0.4330
Test satisfaction ratio:  0.4329

Saved: reports/preprocessing/train_test_split.json


In [12]:
# Step 6b: Impute missing values AFTER split (no data leakage)
# Median computed on TRAIN only, applied to both train and test
imputation_values = {}
for col in X_train.columns:
    if X_train[col].isnull().sum() > 0:
        median_val = X_train[col].median()
        X_train.loc[:, col] = X_train[col].fillna(median_val)
        X_test.loc[:, col] = X_test[col].fillna(median_val)
        imputation_values[col] = float(median_val)

print("\n" + "="*60)
print("MISSING VALUE IMPUTATION (POST-SPLIT — No Data Leakage)")
print("="*60)
print(f"\nStrategy: Median Imputation (fit on TRAIN only)")
print(f"Columns imputed: {list(imputation_values.keys())}")
print(f"Imputation values (from train): {imputation_values}")
print(f"X_train missing after: {X_train.isnull().sum().sum()}")
print(f"X_test missing after:  {X_test.isnull().sum().sum()}")

# Save missing value handling as JSON
missing_value_handling = {
    "strategy": "median_imputation",
    "data_leakage_prevention": "Median computed on TRAINING set only, applied to both train and test",
    "columns_imputed": list(imputation_values.keys()),
    "imputation_values": imputation_values,
    "missing_before": {col: int(val) for col, val in missing_before.items() if val > 0},
    "missing_after_train": int(X_train.isnull().sum().sum()),
    "missing_after_test": int(X_test.isnull().sum().sum()),
    "total_values_imputed": int(sum(missing_before[col] for col in imputation_values.keys()))
}

with open(f"{reports_dir}/missing_value_handling.json", "w") as f:
    json.dump(missing_value_handling, f, indent=2)

print(f"\nSaved: {reports_dir}/missing_value_handling.json")


MISSING VALUE IMPUTATION (POST-SPLIT — No Data Leakage)

Strategy: Median Imputation (fit on TRAIN only)
Columns imputed: ['Arrival Delay in Minutes']
Imputation values (from train): {'Arrival Delay in Minutes': 0.0}
X_train missing after: 0
X_test missing after:  0

Saved: reports/preprocessing/missing_value_handling.json


In [13]:
# Step 7: Apply StandardScaler (fit on train, transform both)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
feature_names = X_train.columns.tolist()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*60)
print("FEATURE SCALING (StandardScaler)")
print("="*60)
print(f"\nScaler type: StandardScaler")
print(f"Features scaled: {len(feature_names)}")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

# Show sample statistics after scaling
print(f"\nSample scaled feature statistics (first 5 features):")
for i, col in enumerate(feature_names[:5]):
    print(f"  {col}: mean={scaler.mean_[i]:.4f}, std={scaler.scale_[i]:.4f}")

# Save feature scaling as JSON
feature_scaling = {
    "scaler_type": "StandardScaler",
    "features_scaled": feature_names,
    "scaler_mean": {col: round(float(scaler.mean_[i]), 6) for i, col in enumerate(feature_names)},
    "scaler_std": {col: round(float(scaler.scale_[i]), 6) for i, col in enumerate(feature_names)},
    "X_train_scaled_shape": list(X_train_scaled.shape),
    "X_test_scaled_shape": list(X_test_scaled.shape)
}

with open(f"{reports_dir}/feature_scaling.json", "w") as f:
    json.dump(feature_scaling, f, indent=2)

print(f"\nSaved: {reports_dir}/feature_scaling.json")


FEATURE SCALING (StandardScaler)

Scaler type: StandardScaler
Features scaled: 23
X_train_scaled shape: (79123, 23)
X_test_scaled shape:  (19781, 23)

Sample scaled feature statistics (first 5 features):
  Age: mean=39.3552, std=15.1261
  Flight Distance: mean=1190.1172, std=996.9519
  Inflight wifi service: mean=2.7336, std=1.3280
  Departure/Arrival time convenient: mean=3.0624, std=1.5252
  Ease of Online booking: mean=2.7610, std=1.3990

Saved: reports/preprocessing/feature_scaling.json


## 7. Preprocessing Summary

In [14]:
# Final preprocessing summary
json_files = [
    "dataset_overview.json",
    "data_quality_report.json",
    "column_cleanup.json",
    "missing_value_handling.json",
    "target_encoding.json",
    "feature_target_split.json",
    "categorical_encoding.json",
    "train_test_split.json",
    "feature_scaling.json",
    "preprocessing_summary.json"
]

preprocessing_steps = [
    "1. Loaded dataset from dataset/test.csv",
    "2. Assessed data quality (missing values, duplicates, dtypes)",
    "3. Dropped unnecessary columns (id, Unnamed: 0)",
    "4. Imputed missing values using median strategy",
    "5. Encoded target variable (satisfaction: 0/1)",
    "6. Separated features (X) and target (y)",
    "7. One-hot encoded categorical features (Gender, Customer Type, Type of Travel, Class)",
    "8. Split data into train/test sets (80/20, stratified)",
    "9. Applied StandardScaler (fit on train, transform both)"
]

print("\n" + "="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"\nPreprocessing date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset: test.csv")
print(f"\nSteps performed:")
for step in preprocessing_steps:
    print(f"  {step}")

print(f"\nOriginal shape: {df_original.shape}")
print(f"Final train shape: {X_train_scaled.shape}")
print(f"Final test shape:  {X_test_scaled.shape}")
print(f"\nTotal JSON files generated: {len(json_files)}")
for jf in json_files:
    print(f"  - {reports_dir}/{jf}")

# Save preprocessing summary as JSON
preprocessing_summary = {
    "preprocessing_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "dataset": "test.csv",
    "total_steps": len(preprocessing_steps),
    "steps": preprocessing_steps,
    "original_shape": list(df_original.shape),
    "final_train_shape": list(X_train_scaled.shape),
    "final_test_shape": list(X_test_scaled.shape),
    "total_json_files_generated": len(json_files),
    "json_files_list": json_files
}

with open(f"{reports_dir}/preprocessing_summary.json", "w") as f:
    json.dump(preprocessing_summary, f, indent=2)


PREPROCESSING SUMMARY

Preprocessing date: 2026-02-15 02:31:39
Dataset: test.csv

Steps performed:
  1. Loaded dataset from dataset/test.csv
  2. Assessed data quality (missing values, duplicates, dtypes)
  3. Dropped unnecessary columns (id, Unnamed: 0)
  4. Imputed missing values using median strategy
  5. Encoded target variable (satisfaction: 0/1)
  6. Separated features (X) and target (y)
  7. One-hot encoded categorical features (Gender, Customer Type, Type of Travel, Class)
  8. Split data into train/test sets (80/20, stratified)
  9. Applied StandardScaler (fit on train, transform both)

Original shape: (98904, 25)
Final train shape: (79123, 23)
Final test shape:  (19781, 23)

Total JSON files generated: 10
  - reports/preprocessing/dataset_overview.json
  - reports/preprocessing/data_quality_report.json
  - reports/preprocessing/column_cleanup.json
  - reports/preprocessing/missing_value_handling.json
  - reports/preprocessing/target_encoding.json
  - reports/preprocessing/fe

## 8. Save Preprocessed Data for Model Training

In [15]:
import pickle

# Save preprocessed data so model_training.ipynb can load it directly
preprocessed_dir = "dataset/preprocessed"
os.makedirs(preprocessed_dir, exist_ok=True)

# Save train/test splits
with open(f"{preprocessed_dir}/X_train_scaled.pkl", "wb") as f:
    pickle.dump(X_train_scaled, f)

with open(f"{preprocessed_dir}/X_test_scaled.pkl", "wb") as f:
    pickle.dump(X_test_scaled, f)

with open(f"{preprocessed_dir}/X_train.pkl", "wb") as f:
    pickle.dump(X_train, f)

with open(f"{preprocessed_dir}/X_test.pkl", "wb") as f:
    pickle.dump(X_test, f)

with open(f"{preprocessed_dir}/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

with open(f"{preprocessed_dir}/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

# Save scaler and feature columns to models/
os.makedirs("models", exist_ok=True)

with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("models/feature_columns.pkl", "wb") as f:
    pickle.dump(X_train.columns, f)

print("Preprocessed data saved to dataset/preprocessed/:")
print(f"  - X_train_scaled.pkl  {X_train_scaled.shape}")
print(f"  - X_test_scaled.pkl   {X_test_scaled.shape}")
print(f"  - X_train.pkl         {X_train.shape}")
print(f"  - X_test.pkl          {X_test.shape}")
print(f"  - y_train.pkl         {y_train.shape}")
print(f"  - y_test.pkl          {y_test.shape}")
print(f"\nModels directory:")
print(f"  - models/scaler.pkl")
print(f"  - models/feature_columns.pkl")
print(f"\nRun model_training.ipynb next to train models using this data.")

Preprocessed data saved to dataset/preprocessed/:
  - X_train_scaled.pkl  (79123, 23)
  - X_test_scaled.pkl   (19781, 23)
  - X_train.pkl         (79123, 23)
  - X_test.pkl          (19781, 23)
  - y_train.pkl         (79123,)
  - y_test.pkl          (19781,)

Models directory:
  - models/scaler.pkl
  - models/feature_columns.pkl

Run model_training.ipynb next to train models using this data.
